# Experimento C: Validación de Integridad Metodológica (The Anti-Leakage Test)

## Objetivo
Demostrar empíricamente cómo la literatura falla al aplicar transformaciones antes de la división de datos.  
Este experimento **valida la contribución metodológica** del TFM.

## Diseño Experimental: Dos Ramas
| Aspecto | Rama "Correcta" (Nuestra propuesta) | Rama "Incorrecta" (Simulación de Leakage) |
|---------|--------------------------------------|-------------------------------------------|
| **División** | Temporal estricta **antes** de cualquier transformación | Se aplica **después** de SMOTE y escalado global |
| **Escalado** | `StandardScaler` solo en Train, proyectar en Test | `StandardScaler` sobre todo el dataset |
| **SMOTE** | Solo dentro del conjunto Train | Sobre todo el dataset (mezcla Train+Test) |
| **Modelo** | Random Forest (100 árboles, seed=42) | Mismo modelo |

## Resultado Esperado
- La rama **"Incorrecta"** dará resultados sospechosamente altos (AUPRC > 0.95)
- La **"Correcta"** dará resultados realistas (AUPRC ~0.80)

## Métricas
- **AUPRC** (Area Under Precision-Recall Curve)
- **AUC ROC**
- **Card Precision@100** (solo rama correcta — la incorrecta no tiene estructura temporal)

## Valor para el TFM
Este gráfico será la **evidencia visual** de la crítica al "Estado del Arte" deficiente.  
Se comparará además con los resultados del **Experimento A (Baseline puro)** para contextualizar la magnitud de la inflación.

In [ ]:
import os
import sys
import warnings
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn import metrics
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Configuración del proyecto
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))

from experiments.config import (
    SEED, INPUT_FEATURES, OUTPUT_FEATURE,
    RESULTS_DIR, FIGURES_DIR, COLORS, TOP_K_LIST,
    START_DATE_TRAINING, DELTA_TRAIN, DELTA_DELAY, DELTA_TEST,
)
from experiments.data_utils import (
    load_transformed_data, get_train_test_set,
    print_dataset_summary, card_precision_top_k,
)

warnings.filterwarnings('ignore')
sns.set_style('darkgrid', {'axes.facecolor': '0.9'})

print("=" * 60)
print("  EXPERIMENTO C: ANTI-LEAKAGE TEST")
print("=" * 60)
print(f"  Semilla: {SEED}")
print(f"  Modelo: Random Forest (n_estimators=100)")
print(f"  Métricas: AUPRC, AUC ROC, Card Precision@100")

---
## 1. Carga de Datos

In [ ]:
transactions_df = load_transformed_data()
print(f"Dataset total: {len(transactions_df):,} transacciones")

---
## 2. Rama CORRECTA: Split temporal primero, luego transformar

In [ ]:
# =============================================================================
# RAMA CORRECTA: División temporal → SMOTE solo en train → Evaluar en test
# =============================================================================
# Protocolo:
#   1. Dividir primero por fecha (train < delay < test)
#   2. StandardScaler ajustado SOLO en train
#   3. SMOTE aplicado SOLO en train
#   4. Entrenar con train aumentado, predecir en test original
# =============================================================================

train_df, test_df = get_train_test_set(
    transactions_df,
    start_date_training=START_DATE_TRAINING,
    delta_train=DELTA_TRAIN,
    delta_delay=DELTA_DELAY,
    delta_test=DELTA_TEST,
)
print_dataset_summary(train_df, test_df, "Rama CORRECTA (sin leakage)")

# Pipeline con imblearn: SMOTE solo se aplica durante fit (train)
correct_pipeline = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=SEED)),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)),
])

correct_pipeline.fit(train_df[INPUT_FEATURES], train_df[OUTPUT_FEATURE])
y_pred_correct = correct_pipeline.predict_proba(test_df[INPUT_FEATURES])[:, 1]

# Métricas estándar
auprc_correct = metrics.average_precision_score(test_df[OUTPUT_FEATURE], y_pred_correct)
auc_correct = metrics.roc_auc_score(test_df[OUTPUT_FEATURE], y_pred_correct)

# Card Precision@100 (requiere estructura temporal: CUSTOMER_ID + TX_TIME_DAYS)
predictions_correct_df = test_df.copy()
predictions_correct_df['predictions'] = y_pred_correct
_, _, cp100_correct = card_precision_top_k(predictions_correct_df, top_k=100)

print(f"\n  Rama CORRECTA (sin leakage):")
print(f"    AUC ROC:        {auc_correct:.4f}")
print(f"    AUPRC:          {auprc_correct:.4f}  ← resultado REALISTA")
print(f"    CP@100:         {cp100_correct:.4f}")

---
## 3. Rama INCORRECTA: Transformar todo el dataset antes de dividir (Data Leakage)

In [ ]:
# =============================================================================
# RAMA INCORRECTA: Transformar TODO → luego dividir (simulación de Leakage)
# =============================================================================
# Este es el error metodológico que cometen muchos papers:
#
# FUENTE DE LEAKAGE 1: StandardScaler.fit_transform(TODO_EL_DATASET)
#   → El scaler ve la media/desviación del futuro (test), contaminando train
#
# FUENTE DE LEAKAGE 2: SMOTE.fit_resample(TODO_EL_DATASET)
#   → Genera muestras sintéticas interpolando entre datos de train y test
#   → Los vecinos cercanos pueden ser transacciones del periodo de test
#
# FUENTE DE LEAKAGE 3: train_test_split aleatorio (no temporal)
#   → Transacciones futuras aparecen en train, pasadas en test
#   → Se pierde la causalidad temporal propia de la detección de fraude
# =============================================================================

# Paso 1: Escalar TODO el dataset (LEAKAGE: scaler ve estadísticas de test)
scaler_global = StandardScaler()
X_all_scaled = scaler_global.fit_transform(transactions_df[INPUT_FEATURES])
y_all = transactions_df[OUTPUT_FEATURE].values
print(f"Dataset original: {len(X_all_scaled):,} transacciones")

# Paso 2: SMOTE sobre TODO (LEAKAGE: genera sintéticos con info de test)
smote = SMOTE(random_state=SEED)
X_resampled, y_resampled = smote.fit_resample(X_all_scaled, y_all)
n_synthetic = len(X_resampled) - len(X_all_scaled)
print(f"Después de SMOTE global: {len(X_resampled):,} muestras (+{n_synthetic:,} sintéticas)")

# Paso 3: Dividir DESPUÉS (LEAKAGE: split aleatorio, no temporal)
X_train_leak, X_test_leak, y_train_leak, y_test_leak = train_test_split(
    X_resampled, y_resampled,
    test_size=0.3,
    random_state=SEED,
    stratify=y_resampled,
)
print(f"Train: {len(X_train_leak):,} | Test: {len(X_test_leak):,}")

# Paso 4: Entrenar y evaluar
clf_incorrect = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
clf_incorrect.fit(X_train_leak, y_train_leak)
y_pred_incorrect = clf_incorrect.predict_proba(X_test_leak)[:, 1]

# Métricas (CP@100 NO se puede calcular: no hay CUSTOMER_ID ni TX_TIME_DAYS)
auprc_incorrect = metrics.average_precision_score(y_test_leak, y_pred_incorrect)
auc_incorrect = metrics.roc_auc_score(y_test_leak, y_pred_incorrect)

y_pred_class_incorrect = (y_pred_incorrect >= 0.5).astype(int)
recall_incorrect = metrics.recall_score(y_test_leak, y_pred_class_incorrect)
precision_incorrect = metrics.precision_score(y_test_leak, y_pred_class_incorrect, zero_division=0)

print(f"\n  Rama INCORRECTA (con Data Leakage):")
print(f"    AUC ROC:        {auc_incorrect:.4f}")
print(f"    AUPRC:          {auprc_incorrect:.4f}  ← resultado INFLADO artificialmente")
print(f"    Recall:         {recall_incorrect:.4f}")
print(f"    Precision:      {precision_incorrect:.4f}")
print(f"    CP@100:         N/A (datos sintéticos sin estructura temporal)")

---
## 4. Visualizaciones Comparativas

In [ ]:
# =============================================================================
# GRÁFICO PRINCIPAL: Barras lado a lado (AUPRC y AUC ROC)
# Requisito Planning Semana 2: "gráfica de barras que ponga lado a lado
# el rendimiento con Data Leakage vs. Sin Data Leakage"
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# --- Panel 1: Barras comparativas (AUPRC + AUC ROC) ---
ax = axes[0]
metric_names = ['AUPRC', 'AUC ROC']
correct_vals = [auprc_correct, auc_correct]
incorrect_vals = [auprc_incorrect, auc_incorrect]

x_pos = np.arange(len(metric_names))
width = 0.3

bars_correct = ax.bar(
    x_pos - width/2, correct_vals, width,
    label='Correcta (sin Leakage)', color=COLORS['correct_pipeline'],
    edgecolor='black', linewidth=0.5,
)
bars_incorrect = ax.bar(
    x_pos + width/2, incorrect_vals, width,
    label='Incorrecta (con Leakage)', color=COLORS['incorrect_pipeline'],
    edgecolor='black', linewidth=0.5,
)

for bar in bars_correct:
    ax.annotate(f'{bar.get_height():.3f}',
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                ha='center', va='bottom', fontsize=11, fontweight='bold')
for bar in bars_incorrect:
    ax.annotate(f'{bar.get_height():.3f}',
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                ha='center', va='bottom', fontsize=11, fontweight='bold',
                color=COLORS['incorrect_pipeline'])

ax.set_ylabel('Valor', fontsize=12)
ax.set_title('Impacto del Data Leakage\nen las Métricas Principales', fontsize=14)
ax.set_xticks(x_pos)
ax.set_xticklabels(metric_names, fontsize=12)
ax.legend(fontsize=10, loc='lower right')
ax.set_ylim([0, 1.15])

# --- Panel 2: Curva PR - Rama CORRECTA ---
ax = axes[1]
prec_c, rec_c, _ = metrics.precision_recall_curve(
    test_df[OUTPUT_FEATURE], y_pred_correct
)
ax.plot(rec_c, prec_c, color=COLORS['correct_pipeline'], linewidth=2.5,
        label=f'Correcta (AP={auprc_correct:.3f})')
baseline_rate = test_df[OUTPUT_FEATURE].mean()
ax.axhline(y=baseline_rate, color='gray', linestyle=':', alpha=0.7,
           label=f'Aleatorio ({baseline_rate:.3f})')
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Curva PR — Rama Correcta\n(división temporal + SMOTE en train)', fontsize=13)
ax.legend(fontsize=10)
ax.set_xlim([0, 1.01]); ax.set_ylim([0, 1.01])

# --- Panel 3: Curva PR - Rama INCORRECTA ---
ax = axes[2]
prec_i, rec_i, _ = metrics.precision_recall_curve(y_test_leak, y_pred_incorrect)
ax.plot(rec_i, prec_i, color=COLORS['incorrect_pipeline'], linewidth=2.5,
        linestyle='--', label=f'Incorrecta (AP={auprc_incorrect:.3f})')
ax.axhline(y=0.5, color='gray', linestyle=':', alpha=0.7,
           label='Aleatorio (0.500)')
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Curva PR — Rama Incorrecta\n(SMOTE global + split aleatorio)', fontsize=13)
ax.legend(fontsize=10)
ax.set_xlim([0, 1.01]); ax.set_ylim([0, 1.01])
ax.annotate('⚠ Inflado por\nData Leakage',
            xy=(0.5, 0.9), fontsize=12, color=COLORS['incorrect_pipeline'],
            ha='center', fontweight='bold')

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'experiment_c_leakage_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Figura guardada: {FIGURES_DIR / 'experiment_c_leakage_comparison.png'}")

---
## 5. Tabla Comparativa: Correcto vs Incorrecto vs Experimento A

Requisito de la **Semana 1** del planning:  
> *"Comparar preliminarmente los resultados de A (Realista) vs C-Incorrecto (Inflado)"*  
> *"Elaborar tabla comparativa A vs C-Incorrecto"*

In [ ]:
# ---------------------------------------------------------------
# Tabla 1: Resultado directo del Experimento C (Correcto vs Incorrecto)
# ---------------------------------------------------------------
comparison_c = pd.DataFrame({
    'Pipeline': [
        'C-Correcta (temporal + SMOTE en train)',
        'C-Incorrecta (SMOTE global + split aleatorio)',
    ],
    'AUC ROC': [auc_correct, auc_incorrect],
    'AUPRC': [auprc_correct, auprc_incorrect],
    'CP@100': [cp100_correct, np.nan],  # N/A para incorrecta
}).set_index('Pipeline')

print("\n" + "=" * 80)
print("  Tabla 1: Resultado del Experimento C")
print("=" * 80)
display(comparison_c.round(4))

# ---------------------------------------------------------------
# Tabla 2: Comparativa A (Baseline puro) vs C-Incorrecto (Inflado)
# Requisito del planning Semana 1
# ---------------------------------------------------------------
exp_a_path = RESULTS_DIR / 'experiment_a_predictions.pkl'

if exp_a_path.exists():
    with open(exp_a_path, 'rb') as f:
        results_a = pickle.load(f)

    # Construir tabla: A (cada modelo) + C-Correcto + C-Incorrecto
    rows = []
    for model_name, res_a in results_a.items():
        cp100_a = res_a.get('card_precision_at_k', {}).get(100, np.nan)
        rows.append({
            'Experimento': f'A-Baseline: {model_name}',
            'AUC ROC': res_a['auc_roc'],
            'AUPRC': res_a['avg_precision'],
            'CP@100': cp100_a,
        })

    rows.append({
        'Experimento': 'C-Correcta (SMOTE en train)',
        'AUC ROC': auc_correct,
        'AUPRC': auprc_correct,
        'CP@100': cp100_correct,
    })
    rows.append({
        'Experimento': 'C-Incorrecta (con Leakage) ⚠',
        'AUC ROC': auc_incorrect,
        'AUPRC': auprc_incorrect,
        'CP@100': np.nan,
    })

    comparison_a_vs_c = pd.DataFrame(rows).set_index('Experimento')

    print("\n" + "=" * 80)
    print("  Tabla 2: Comparativa A (Realista) vs C-Incorrecto (Inflado)")
    print("=" * 80)
    display(comparison_a_vs_c.round(4))

    # Guardar
    comparison_a_vs_c.to_csv(RESULTS_DIR / 'experiment_c_vs_a_comparison.csv')
    print(f"\n✓ Tabla comparativa A vs C: {RESULTS_DIR / 'experiment_c_vs_a_comparison.csv'}")
else:
    print("\n⚠ No se encontraron resultados del Experimento A.")
    print(f"  Ruta esperada: {exp_a_path}")
    print("  Ejecuta primero experiment_a_baseline.ipynb para generar la comparativa.")

# Guardar tabla básica de C
comparison_c.to_csv(RESULTS_DIR / 'experiment_c_comparison.csv')
print(f"✓ Tabla Exp C: {RESULTS_DIR / 'experiment_c_comparison.csv'}")

---
## 6. Persistencia de Resultados

In [ ]:
# Guardar resultados completos del Experimento C
results_c = {
    'correct': {
        'auc_roc': auc_correct,
        'auprc': auprc_correct,
        'card_precision_at_100': cp100_correct,
        'y_pred_proba_test': y_pred_correct,
    },
    'incorrect': {
        'auc_roc': auc_incorrect,
        'auprc': auprc_incorrect,
        'recall': recall_incorrect,
        'precision': precision_incorrect,
        'y_pred_proba_test': y_pred_incorrect,
    },
    'metadata': {
        'model': 'RandomForest(n=100)',
        'seed': SEED,
        'leakage_sources': [
            'StandardScaler global (media/std contaminada)',
            'SMOTE global (sintéticos con info de test)',
            'train_test_split aleatorio (no temporal)',
        ],
    },
}

with open(RESULTS_DIR / 'experiment_c_results.pkl', 'wb') as f:
    pickle.dump(results_c, f)

print("✓ Resultados del Experimento C guardados exitosamente")
print(f"  - PKL: {RESULTS_DIR / 'experiment_c_results.pkl'}")
print(f"  - CSV: {RESULTS_DIR / 'experiment_c_comparison.csv'}")
print(f"  - FIG: {FIGURES_DIR / 'experiment_c_leakage_comparison.png'}")

---
## 7. Conclusiones del Experimento C

### Evidencia del Data Leakage

**Este experimento es la evidencia visual de la crítica al "Estado del Arte" deficiente.**

La diferencia entre ambas ramas demuestra cómo el Data Leakage infla artificialmente los resultados, dando una **falsa sensación de seguridad** sobre el rendimiento del modelo.

### Fuentes de Leakage Identificadas

1. **Escalado global**: `StandardScaler` ajustado con datos de test → la media y desviación incluyen información futura
2. **SMOTE global**: Las muestras sintéticas se generan interpolando vecinos que pueden pertenecer al periodo de test
3. **Split aleatorio**: Rompe la causalidad temporal, permitiendo que transacciones futuras entrenen al modelo

### Implicación para el TFM

Los resultados "inflados" de la rama incorrecta son comparables a los que publica parte de la literatura sobre detección de fraude.  
La rama correcta produce resultados **realistas**, coherentes con los del Experimento A (Baseline puro), y sirve como **validación de la integridad metodológica** del TFM.

### Relación con otros experimentos

- **vs Experimento A**: Los resultados de C-Correcto deberían ser ligeramente superiores a A (gracias a SMOTE en train), pero del mismo orden de magnitud.
- **vs C-Incorrecto**: La diferencia cuantifica el "coste" del leakage en la evaluación del modelo.